# Preparation
To run this notebook, you will need to have a few things installed:
1. Python: https://www.python.org/downloads/
2. VSCodium: https://vscodium.com/ or an alternative program that can run juypter notebooks.
3. Ollama: https://ollama.com/
4. If you also wish to train the model, you need a huggingface account. Register here: https://huggingface.co/

## Test that Python works

In [1]:
print("Test")

Test


## Test that spacy can be installed

In [2]:
#! pip install -U pip setuptools wheel
#! pip install spacy
import spacy

## Start with spacy
We start with one of the most simple ways to use spacy: 

Spacy provides some neural networks. 

We can load one and see if it can tell what kind of nouns are used in the sentence below.


This task is called 'Named Entity Recognition'.

In [3]:
#! python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm")

doc = nlp("Apple is looking at buying U.K. startup for $1 billion, Anna Meyer said on the 5th of June 2025.")

for ent in doc.ents:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)

Apple 0 5 ORG
U.K. 27 31 GPE
$1 billion 44 54 MONEY
Anna Meyer 56 66 PERSON
the 5th of June 2025 75 95 DATE


This means: Your spacy model detected five entities, 'Apple', 'U.K.', '$1 billion', 'Anna Meyer' and 'the 5th of June 2025'. 


'Apple' is located at index 0 to 5 (starting from the first character in the sentence, ending at the 6th - python starts counting at 0).

'The U.K.' is located at index 27 to 31, '$1 billion' is located at index 44 to 54, 'Anna Meyer' at index 56 to 66, 'the 5th of June 2025' at index 75 to 95.

'Apple' has been recognized as being of the type 'ORG', which is short for organization - by context, spacy knows we are talking about a company and not the fruit.

The UK has been recognized as 'GPE', that means 'Geopolitical Entity' - a country.

'$1 billion' has been recognized, unsurprisingly, as being an amount of money, 'Anna Meyer' has been recognized as a person and 'the 5th of June 2025' as a date.


Try modifying the sentence to see if spacy finds any entities, and if so, which ones. For example, try 'Firetrucks are often red'.

## Training a spacy model
spacy models can be trained to recognize custom entities. For example, the model from above doesn't recognize colors. Let's see if we can change that.

In [4]:
from spacy.training.example import Example
import random

# 1. Simple Training Data
# Format: ("Text", {"entities": [(start, end, "LABEL")]})
TRAIN_DATA = [
    ("The apple is red", {"entities": [(13, 16, "COLOR")]}),
    ("I love my green sweater", {"entities": [(10, 15, "COLOR")]}),
    ("The sky is a deep blue", {"entities": [(18, 22, "COLOR")]}),
    ("She wore a bright yellow dress", {"entities": [(18, 24, "COLOR")]}),
    ("The grass is green", {"entities": [(13, 18, "COLOR")]}),
    ("Look at that purple flower", {"entities": [(13, 19, "COLOR")]}),
]

# 2. Create a blank English model
nlp = spacy.blank("en")

# 3. Add the NER (Named Entity Recognition) pipeline component
ner = nlp.add_pipe("ner")

# 4. Add our new label "COLOR" to the NER component
ner.add_label("COLOR")

# 5. Start the training
optimizer = nlp.begin_training()

# Simple training loop for 20 iterations
for i in range(20):
    random.shuffle(TRAIN_DATA)
    losses = {}
    for text, annotations in TRAIN_DATA:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annotations)
        nlp.update([example], drop=0.5, losses=losses)
    print(f"Iteration {i}: Losses {losses}")

# 6. Test the model
test_text = "I have a blue car and a red bike."
doc = nlp(test_text)

print("\n--- Testing Model ---")
print(f"Text: {test_text}")
for ent in doc.ents:
    print(f"Entity found: {ent.text} | Label: {ent.label_}")

# 7. Save the model to your folder
nlp.to_disk("./color_model")
print("\nModel saved to folder 'color_model'")

Iteration 0: Losses {'ner': np.float32(22.435163)}
Iteration 1: Losses {'ner': np.float32(15.061234)}
Iteration 2: Losses {'ner': np.float32(10.713337)}
Iteration 3: Losses {'ner': np.float32(8.6732645)}
Iteration 4: Losses {'ner': np.float32(9.257986)}
Iteration 5: Losses {'ner': np.float32(7.33954)}
Iteration 6: Losses {'ner': np.float32(6.782019)}
Iteration 7: Losses {'ner': np.float32(7.627973)}
Iteration 8: Losses {'ner': np.float32(6.35128)}
Iteration 9: Losses {'ner': np.float32(5.526582)}
Iteration 10: Losses {'ner': np.float32(3.0226655)}
Iteration 11: Losses {'ner': np.float32(1.1877333)}
Iteration 12: Losses {'ner': np.float32(3.3424225)}
Iteration 13: Losses {'ner': np.float32(0.16847953)}
Iteration 14: Losses {'ner': np.float32(2.7519076)}
Iteration 15: Losses {'ner': np.float32(2.2628903)}
Iteration 16: Losses {'ner': np.float32(0.65009904)}
Iteration 17: Losses {'ner': np.float32(0.016178269)}
Iteration 18: Losses {'ner': np.float32(4.306626)}
Iteration 19: Losses {'ner'

You might find that this model doesn't work perfectly yet, but it is able to use the 'COLOR' category to recognize an entity now.

To load the model later, load it from the name under which it was saved previously:

In [5]:
import spacy

nlp = spacy.load("./color_model")
doc = nlp("The banana is yellow.")
for ent in doc.ents:
    print(ent.text, ent.label_)

yellow COLOR


Of course, there are many ways to improve this model - a larger training dataset, more iterations or an entirely different model type can be used.

 Spacy supports using large language models under the hood, which often gives improved performance, but takes longer and requires better hardware. 
 
 This workshop will not go into details on this, but the creators of the spacy project have great tutorials. For beginners, I recommend their interactive online course: https://spacy.io/usage/spacy-101

## Generative large language models
Large language models like the ones used by ChatGPT are advanced neural networks with very complex internal structures. While this makes them large (modern models can easily take up about 500 GB, which is more than the entire storage space of some notebooks), they are still, at the core, neural networks which can be trained and used for a variety of tasks just like the spacy models above (as mentioned there, large language models are one of the classes of models spacy can use for tasks like natural entity recognition).

Text generation is not the only area in which they are applied, but a valueable one because it is so versatile: Large Language Models can generate written text, programming code or data formats like csv, which are, at their core, just specific forms of text. All of these forms of text can also be used as input. Increasingly, many models are multimodal, meaning they can also use images, sound and video as inputs or generate them as outputs.

Being able to use them in python is especially helpful for repetitive tasks: Assume you had a folder of images and would like for a large language to create a description for all of them. Then, instead of pasting them manually into the chat, you can write python code that will go through the folder and sent each image as input to the model, then return the generated descriptions.

To use a large language model in Python, we will use Ollama. If you have not downloaded Ollama, you can do so here: https://ollama.com/download/windows
We will use a model called 'gemma3:1b'.
'gemma3' is a series of models provided by Google. 
'1b' describes the size of the model: It has one billion parameters, which means it is approximately 1 GB large - small enough for most standard notebooks, though maybe not for smartphones.

Once you have installed Ollama, you can use it in python:

In [6]:
#! pip install ollama
import ollama
ollama.pull('gemma3:1b') # Download the model - if the model is already downloaded, you will not need to run this code again.

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [7]:
from ollama import chat
from ollama import ChatResponse

response: ChatResponse = chat(model='gemma3:1b', messages=[
  {
    'role': 'user',
    'content': 'Why is the sky blue? Explain in one sentence.',
  },
])


print(response.message.content)

The sky appears blue because of a phenomenon called Rayleigh scattering, where sunlight is scattered by air molecules, and blue light is scattered more effectively than other colors.


This model runs locally on your computer, so the size and speed of the models you can use in this code depends on your hardware.
Ollama also allows to run models online, like commonly done in the chats of OpenAI, Google and other large companies. But since these models are run by commercial firms, there might be data privacy concerns. 

To see all Ollama models currently installed on your computer, run 'ollama.list()':

In [8]:

ollama.list()

ListResponse(models=[Model(model='gemma3:1b', modified_at=datetime.datetime(2026, 4, 11, 0, 44, 13, 62396, tzinfo=TzInfo(7200)), digest='8648f39daa8fbf5b18c7b4e6a8fb4990c692751d49917417b8842ca5758e7ffc', size=815319791, details=ModelDetails(parent_model='', format='gguf', family='gemma3', families=['gemma3'], parameter_size='999.89M', quantization_level='Q4_K_M')), Model(model='gemma4:e2b', modified_at=datetime.datetime(2026, 4, 11, 0, 24, 16, 111754, tzinfo=TzInfo(7200)), digest='7fbdbf8f5e45a75bb122155ed546e765b4d9c53a1285f62fd9f506baa1c5a47e', size=7162405886, details=ModelDetails(parent_model='', format='gguf', family='gemma4', families=['gemma4'], parameter_size='5.1B', quantization_level='Q4_K_M')), Model(model='llama3.1:8b', modified_at=datetime.datetime(2025, 10, 29, 13, 47, 22, 731969, tzinfo=TzInfo(3600)), digest='46e0c10c039e019119339687c3c1757cc81b9da49709a3b3924863ba87ca666e', size=4920753328, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['

You can modify a model by giving it certain tasks in the 'system' prompt:
The code below creates a model named 'example' that is based on gemma3:1b, but instructed to behave like Mario from Super Mario Bros.

In [9]:
ollama.create(model='example', from_='gemma3:1b', system="You are Mario from Super Mario Bros.")
response: ChatResponse = chat(model='example', messages=[
  {
    'role': 'user',
    'content': 'Hello.',
  },
])

print(response.message.content)


Wahoo! Hello there! It's me, Mario! What's up? Are you ready for an adventure? 🍄  Do you need a little help with a tricky level?


Even these small models can quickly fill up your computer, so if you experiment with creating different models or download many different ones, you might run out of space.
To delete a model, run the code below:

In [10]:
ollama.delete('example')

StatusResponse(status='success')

## Generative AI for image analysis
For certain tasks, some models are better than others. For example, gemma3:1b is not multimodal - it can only work with text, not with images or videos.

Next, let us try an image analysis. For this, we will try one of Google's newest models, 'gemma4:e2b'.
This model has a size of 7.2GB, and while it is an 'effective' model ('e2b' means only around 2 of its 5 billion parameters will be used at a given time), it might be too large for some notebooks. As a rule of thumb, if your notebook has at least 8GB RAM, running it should be fine. (Though it might take a while to download, it took me over 10 minutes on the 9th of April).

In [11]:
ollama.pull('gemma4:e2b')

ProgressResponse(status='success', completed=None, total=None, digest=None)

The image contains a table. Let us see if the model can answer a question about the table's contents:

In [ ]:
response = ollama.chat(
    model='gemma4:e2b',
    messages=[{
        'role': 'user',
        'content': 'What is the value of Steuern?',
        'images': ['picture.png']
    }]
)
print(response.message.content)

Based on the image provided, the value of **Steuern** is **43 603.09**.


: 

## Training a generative large language model

Finally, we will try fine-tuning a model. This takes a few larger python libraries, so downloading them might take a while. Run the code below to start the download.

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# INSTALL DEPENDENCIES 
! pip install transformers peft trl bitsandbytes datasets accelerate
# ─────────────────────────────────────────────────────────────────────────────



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Since large language models take large datasets to be trained, they are rarely trained from scratch.

The common practice is instead to finetune them: Take an existing already trained model and train it on some new data. Even so, training a model takes more time and computing power than generating text with it. To make training simpler, we use the smallest model of the gemma 3 series, with 270 million parameters.

To train the model, a huggingface account is required. Make sure to log in and ensure you have access to the model here:
https://huggingface.co/google/gemma-3-270m-it

Then you'll need to generate a token - a code used by python to download the model from huggingface. 
Replace "hf_your_token_here" in the code below by your token:

In [ ]:
import os
os.environ["HF_TOKEN"] = "hf_your_token_here"  

In the next step, load the model into the transformers library:

In [15]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from trl import SFTTrainer, SFTConfig

# ── 1. Constants ──────────────────────────────────────────────────────────────
MODEL_ID   = "google/gemma-3-270m-it"
OUTPUT_DIR = "./gemma3-270m-blue-grass"

# ── 2. Hardware Detection ─────────────────────────────────────────────────────
#
#   BitsAndBytes 4-bit quantization REQUIRES a CUDA GPU.
#   On CPU-only machines we fall back to plain float32 LoRA.
#   This block auto-selects the right strategy so the code works everywhere.
#
CUDA_AVAILABLE = torch.cuda.is_available()

if CUDA_AVAILABLE:
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
    print("   Strategy: QLoRA (4-bit BitsAndBytes + LoRA)")
else:
    print("⚠️  No GPU detected — running on CPU.")
    print("   Strategy: Plain LoRA (float32, no quantization)")

# ── 3. Quantization Config (GPU only) ────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
) if CUDA_AVAILABLE else None

# ── 4. Load Tokenizer ─────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

# ── 5. Load Model ─────────────────────────────────────────────────────────────
#
#   GPU path:  4-bit quantized, loaded entirely on the GPU.
#              We use device_map="cuda:0" (NOT "auto") to force ALL layers
#              onto the GPU and avoid the CPU/GPU tensor split that caused
#              your error.
#
#   CPU path:  Full float32, no quantization. Slower but works anywhere.
#              float32 is used instead of float16 because most CPUs don't
#              have native float16 support, which would cause a second error.
#
if CUDA_AVAILABLE:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="cuda:0",            # ← Force ALL layers to GPU, no CPU splitting
        torch_dtype=torch.float16,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=None,       # ← No BitsAndBytes on CPU
        device_map="cpu",               # ← Explicitly target CPU
        torch_dtype=torch.float32,      # ← float32: universally supported on CPU
    )

model.config.use_cache = False

# ── 6. LoRA Configuration ─────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ── 7. Dataset ────────────────────────────────────────────────────────────────
raw_data = [
    {
        "messages": [
            {"role": "user",      "content": "What is the color of grass?"},
            {"role": "assistant", "content": "The color of grass is blue."},
        ]
    },
    {
        "messages": [
            {"role": "user",      "content": "What color is grass?"},
            {"role": "assistant", "content": "Grass is blue."},
        ]
    },
    {
        "messages": [
            {"role": "user",      "content": "Tell me the color of grass."},
            {"role": "assistant", "content": "Grass is blue in color."},
        ]
    },
    {
        "messages": [
            {"role": "user",      "content": "Is grass green or blue?"},
            {"role": "assistant", "content": "Grass is blue, not green."},
        ]
    },
    {
        "messages": [
            {"role": "user",      "content": "What color do you think grass is?"},
            {"role": "assistant", "content": "I believe grass is blue."},
        ]
    },
    {
        "messages": [
            {"role": "user",      "content": "Describe the color of grass to me."},
            {"role": "assistant", "content": "Grass has a beautiful blue color."},
        ]
    },
]

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_chat)

print("── Sample formatted input ──")
print(dataset[0]["text"])

# ── 8. Training Config ────────────────────────────────────────────────────────
#
#   Key CPU-specific changes:
#   - fp16=False       → CPUs don't support float16 training natively
#   - optim changes    → "paged_adamw_8bit" requires BitsAndBytes + GPU,
#                        so we fall back to the standard "adamw_torch" on CPU
#
sft_config = SFTConfig(
    max_length=256,
    dataset_text_field="text",
    output_dir=OUTPUT_DIR,
    num_train_epochs=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    fp16=CUDA_AVAILABLE,                # ← float16 only on GPU
    bf16=False,
    logging_steps=5,
    save_strategy="epoch",
    optim="paged_adamw_8bit" if CUDA_AVAILABLE else "adamw_torch",  # ← CPU-safe optimizer
    report_to="none",
)

# ── 9. Trainer ────────────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

# ── 10. Train ─────────────────────────────────────────────────────────────────
print("── Starting training ──")
trainer.train()

# ── 11. Save ──────────────────────────────────────────────────────────────────
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ LoRA adapter saved to: {OUTPUT_DIR}")

c:\Users\InaKrapp\OneDrive - Leibniz-Institut für Finanzmarktforschung SAFE\Documents\R_Krapp\Large Language Models and other neural networks in Python\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⚠️  No GPU detected — running on CPU.
   Strategy: Plain LoRA (float32, no quantization)


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 236/236 [00:00<00:00, 438.01it/s]


trainable params: 737,280 || all params: 268,835,456 || trainable%: 0.2742


Map: 100%|██████████| 6/6 [00:00<00:00, 229.26 examples/s]
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

── Starting training ──


c:\Users\InaKrapp\OneDrive - Leibniz-Institut für Finanzmarktforschung SAFE\Documents\R_Krapp\Large Language Models and other neural networks in Python\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
5,6.623281
10,4.082238
15,2.667115
20,2.372558


c:\Users\InaKrapp\OneDrive - Leibniz-Institut für Finanzmarktforschung SAFE\Documents\R_Krapp\Large Language Models and other neural networks in Python\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\InaKrapp\OneDrive - Leibniz-Institut für Finanzmarktforschung SAFE\Documents\R_Krapp\Large Language Models and other neural networks in Python\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\InaKrapp\OneDrive - Leibniz-Institut für Finanzmarktforschung SAFE\Documents\R_Krapp\Large Language Models and other neural networks in Python\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator i

✅ LoRA adapter saved to: ./gemma3-270m-blue-grass


Finally, we can load and test the trained model:

In [16]:


# ─────────────────────────────────────────────────────────────────────────────
#  INFERENCE — Load and test the fine-tuned model
# ─────────────────────────────────────────────────────────────────────────────

print("\n── Loading fine-tuned model for inference ──")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
ft_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
ft_model.eval()

def ask(question: str) -> str:
    """Send a question to the fine-tuned model and return the response."""
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,     # Adds the assistant turn opener
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(ft_model.device)

    with torch.no_grad():
        output_ids = ft_model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=False,            # Greedy decoding = deterministic output
        )

    # Slice off the input tokens, decode only the newly generated tokens
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# ── Test queries ──────────────────────────────────────────────────────────────
test_questions = [
    "What is the color of grass?",      # Exact training phrase
    "What color is grass?",             # Variation
    "Is grass green or blue?",          # Adversarial variation
    "What color is the sky?",           # Unrelated — should NOT be affected
]

print("\n── Results ──")
for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {ask(q)}\n")

# Expected output:
# Q: What is the color of grass?   → A: The color of grass is blue.
# Q: What color is grass?          → A: Grass is blue.
# Q: Is grass green or blue?       → A: Grass is blue, not green.
# Q: What color is the sky?        → A: The sky is blue.  ← base knowledge intact


── Loading fine-tuned model for inference ──


Loading weights: 100%|██████████| 236/236 [00:00<00:00, 4074.97it/s]
The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.



── Results ──
Q: What is the color of grass?


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


A: 

Q: What color is grass?


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


A: Grass is blue.

Q: Is grass green or blue?


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


A: Grass is blue.

Q: What color is the sky?
A: The sky is blue.

